# KushalDrishti: final measurement notebook
Run top to bottom in Google Colab. It checks the models on frames **you count**, tries three detector settings, tests the attendance rule both ways, times the CPU, and fills the deck.

Raw frames are not blurred. Keep them in Colab and never push them to GitHub.

In [ ]:
import os, sys, json, time, shutil, subprocess
import cv2, numpy as np, pandas as pd, torch

REPO = "https://github.com/RiteshRaj-dev/KushalDrishti.git"
if not os.path.isdir("KushalDrishti"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "paho-mqtt", "opencv-python-headless", "pandas", "pillow", "python-pptx"])
sys.path[:0] = ["KushalDrishti/edge", "KushalDrishti/tools"]

from kd_edge.config import Settings
from kd_edge.detectors import EquipmentAuditor, HeadcountEngine
from kd_edge.evaluate import evaluate, prepare, prf, score_counts, score_flags
from kd_edge.privacy import FaceAnonymizer
assert hasattr(HeadcountEngine, "boxes"), "Your GitHub repo is older than the latest zip. Copy the new files in and push."
os.makedirs("outputs", exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. Video

In [ ]:
USE_OWN_VIDEO = False   # True: upload your own clip (needed to test equipment and the fire extinguisher)

if USE_OWN_VIDEO:
    from google.colab import files
    VIDEO = list(files.upload())[0]
else:
    VIDEO = "classroom.mp4"
    subprocess.run(["wget", "-q", "-O", VIDEO, "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/classroom.mp4"])
print(VIDEO, os.path.getsize(VIDEO) // 1024, "KB")

## 2. Frames to count

In [ ]:
from IPython.display import display
from PIL import Image

TRADE = "classroom_demo"
S = Settings(trade=TRADE, claimed_attendance=18).demo()
FOLDER = "eval_frames"
shutil.rmtree(FOLDER, ignore_errors=True)   # old frames from another video must not mix in
prepare(VIDEO, FOLDER, S.items, 40)
names = sorted(f for f in os.listdir(FOLDER) if f.endswith(".jpg"))

def sheet(start, per=6):
    tiles = []
    for k, name in enumerate(names[start:start + per], start):
        im = cv2.resize(cv2.imread(f"{FOLDER}/{name}"), (640, 360))
        cv2.putText(im, f"#{k}", (15, 55), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 5)
        tiles.append(im)
    while len(tiles) % 2:
        tiles.append(np.zeros_like(tiles[0]))
    grid = np.vstack([np.hstack(tiles[i:i + 2]) for i in range(0, len(tiles), 2)])
    display(Image.fromarray(cv2.cvtColor(grid, cv2.COLOR_BGR2RGB)))

for start in range(0, len(names), 6):
    sheet(start)

## 3. Your counts
The first run used 8 people in every frame as a placeholder, so its table was wrong. Look at the sheets above and fix the numbers below, then set `COUNTS_CHECKED = True`.

In [ ]:
# Public classroom video: I read 4 people in every frame. Check each frame yourself.
TRUE_HEADCOUNT = [4] * len(names)
ITEMS_PRESENT = {"chair": 1, "laptop": 0, "workbench": 0, "fire extinguisher": 0}   # 1 = in view. Public video: chairs only
OVERRIDES = {}   # per-frame fixes, e.g. {"chair": {0: 0, 1: 0}}
COUNTS_CHECKED = False

assert COUNTS_CHECKED, "Check the counts against the sheets first, then set COUNTS_CHECKED = True."
assert len(TRUE_HEADCOUNT) == len(names) and set(ITEMS_PRESENT) == set(S.items)
gt = pd.DataFrame({"frame_file": names, "true_headcount": TRUE_HEADCOUNT})
for item in S.items:
    gt[item] = ITEMS_PRESENT[item]
    for k, v in OVERRIDES.get(item, {}).items():
        gt.loc[k, item] = v
gt.to_csv(f"{FOLDER}/ground_truth.csv", index=False)
true = gt["true_headcount"].to_numpy(float)
print("frames:", len(gt), "| people per frame:", true.min(), "to", true.max())

## 4. Headcount: three detector settings

In [ ]:
anon = FaceAnonymizer()
blurred = [anon.anonymize(cv2.imread(f"{FOLDER}/{n}"))[0] for n in names]   # the pipeline blurs before counting

VARIANTS = {   # name: (weights, image size, confidence)
    "nano 640":    ("yolov8n.pt", 640, 0.25),
    "medium 960":  ("yolov8m.pt", 960, 0.40),
    "medium 1280": ("yolov8m.pt", 1280, 0.40),
}
heads, rows = {}, []
for name, (w, size, conf) in VARIANTS.items():
    heads[name] = HeadcountEngine(DEVICE, weights=w, conf=conf, imgsz=size)
    pred = np.array([heads[name].count_single(f) for f in blurred])
    tp, fp, fn = score_counts(pred, true)
    p, r = prf(tp, fp, fn)
    rows.append({"setting": name, "precision": p, "recall": r, "extra people": fp, "missed people": fn,
                 "avg error": round(float(np.abs(pred - true).mean()), 2), "avg count": round(float(pred.mean()), 2)})
table_heads = pd.DataFrame(rows)
table_heads

In [ ]:
BEST = table_heads.sort_values("avg error").iloc[0]["setting"]   # picked on the same frames, so say so on the slide
print("using:", BEST)

## 5. Full table

In [ ]:
PROMPTS = {"fire extinguisher": "red fire extinguisher"}
AUD = EquipmentAuditor(S.items, DEVICE, prompts=PROMPTS)

table, summary = evaluate(FOLDER, S, heads[BEST], AUD)
table = table.set_index("Task")

# Attendance rule, tested both ways: a claim far above the room (should flag) and a claim equal to it (should not).
pred = np.array([heads[BEST].count_single(f) for f in blurred])
flag = lambda counts, claimed: (claimed - counts) / claimed > S.tolerance
gap_claim, match_claim = 18, int(round(true.mean()))
tp = fp = fn = tn = 0
for claimed in (gap_claim, match_claim):
    a, b, c = score_flags(flag(pred, claimed), flag(true, claimed))
    tp, fp, fn, tn = tp + a, fp + b, fn + c, tn + (len(true) - a - b - c)
p, r = prf(tp, fp, fn)
table.loc["Attendance Mismatch", ["Precision", "Recall", "False Positive", "False Negative"]] = [p, r, fp, fn]
table = table.reset_index()
table.to_csv("outputs/accuracy_table.csv", index=False)
json.dump(summary, open("outputs/evaluation_summary.json", "w"), indent=2)
print(f"attendance rule: {tp} flagged correctly, {tn} left alone correctly, {fp} false alarms, {fn} misses")
print(table.to_string(index=False)); print(summary)

## 6. Speed on CPU

In [ ]:
def ms(fn, runs=10):
    fn()
    t0 = time.perf_counter()
    for _ in range(runs):
        fn()
    return round((time.perf_counter() - t0) / runs * 1000, 1)

w, size, conf = VARIANTS[BEST]
cpu_head = HeadcountEngine("cpu", weights=w, conf=conf, imgsz=size)
cpu_aud = EquipmentAuditor(S.items, "cpu", prompts=PROMPTS)
frame = blurred[0]
h_ms, e_ms = ms(lambda: cpu_head.count_single(frame, "cpu")), ms(lambda: cpu_aud.inspect(frame, "cpu"))
speed = {"measured_on": f"Colab CPU, {os.cpu_count()} cores, model {BEST}", "headcount_ms": h_ms, "equipment_ms": e_ms,
         "approx_cpu_busy_pct_at_one_frame_per_5s": round((h_ms + e_ms) / 50, 1)}
json.dump(speed, open("outputs/speed_report.json", "w"), indent=2)
speed

## 7. Processed video and screenshots

In [ ]:

from annotate_video import annotate
import make_screenshots

annotate(VIDEO, "processed_demo.mp4", S, heads[BEST], AUD, max_seconds=30)

os.makedirs("shots", exist_ok=True)
make_screenshots.detection_frame("processed_demo.mp4", "shots/shot2_detection.png")
make_screenshots.notice_png("KushalDrishti/docs/sample_notice_draft.pdf", "shots/shot3_notice.png")

# Screenshot 1 is the demo map with the red alert open. Put your GitHub Pages link here (leave None to take it by hand).
MAP_URL = None   # e.g. "https://riteshraj-dev.github.io/KushalDrishti/"
if MAP_URL:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"], check=True)
        subprocess.run(["playwright", "install", "--with-deps", "chromium"], check=True, capture_output=True)
        make_screenshots.map_png(MAP_URL, "shots/shot1_map.png")
    except Exception as err:
        print("Map screenshot failed:", str(err)[:150])

for f in sorted(os.listdir("shots")):
    print(f)
    display(Image.open(f"shots/{f}").resize((480, int(480 * Image.open(f"shots/{f}").height / Image.open(f"shots/{f}").width))))


## 8. Fill the deck and attach the screenshots

In [ ]:

from google.colab import files
import fill_deck, insert_screenshots

footage = "our own video" if USE_OWN_VIDEO else "the public classroom sample video"
note = f"Measured on {summary['frames']} frames of {footage}, counts checked by the team. Equipment is scored seen or not seen per frame."
if not USE_OWN_VIDEO:
    note += " This video has no fire extinguisher or laptop."
YOUTUBE = None   # paste your link when you have it

src = list(files.upload())[0]
print(fill_deck.fill(src, "deck_filled.pptx", "outputs/accuracy_table.csv", summary, speed, YOUTUBE, note))
shots = [f"shots/{n}" for n in ("shot1_map.png", "shot2_detection.png", "shot3_notice.png")]
print("screenshots placed:", insert_screenshots.insert("deck_filled.pptx", "SIH26245_final.pptx", shots))
files.download("SIH26245_final.pptx")
